# Introducción a la Ciencia de Datos: Tarea 1

Solución basada en el template provisto. Los datos se cargan directamente desde el archivo CSV local.
Dataset: artículos de noticias de distintos medios de prensa para clasificación por fuente.

## Cargar bibliotecas (dependencias)

In [ ]:
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import networkx as nx
import missingno as msno

from src.utils import (
    ORDER,
    PALETTE,
    LABEL_TO_PUB,
    PUB_TO_LABEL,
    STOPWORDS,
    N_TOP_WORDS,
    load_dataset,
    clean_text,
    plot_top_words,
    plot_wordclouds_by_source,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

# Aliases para mantener compatibilidad con el resto del notebook.
label_to_pub = LABEL_TO_PUB
pub_to_label = PUB_TO_LABEL

## Lectura de Datos

Se carga el archivo CSV ya descargado en lugar de utilizar la librería `datasets` de HuggingFace.

In [ ]:
df = load_dataset()
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')

In [ ]:
df.head()

In [ ]:
df.info()

# Parte 1: Cargado y Limpieza de Datos

## A. Exploración de Datos

In [ ]:
# Datos faltantes por columna
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Faltantes': missing, 'Porcentaje (%)': missing_pct})
missing_df = missing_df[missing_df['Faltantes'] > 0].sort_values('Faltantes', ascending=False)
print('Columnas con datos faltantes:')
print(missing_df.to_string())

**Matriz de nulidad e informe de data profiling:**

Para complementar el análisis numérico de datos faltantes, se utilizan dos herramientas visuales:

- `missingno` para generar la matriz de nulidad (Figura 1 del informe), que muestra la distribución de valores nulos por columna a lo largo de las filas del dataset.
- `ydata-profiling` para generar un reporte HTML interactivo con distribuciones, correlaciones y estadísticas detalladas (opcional, requiere recursos).

In [ ]:
# Matriz de nulidad: muestra dónde aparecen los valores faltantes en el dataset.
fig, ax = plt.subplots(figsize=(12, 5))
msno.matrix(df, ax=ax, sparkline=False, fontsize=11)
ax.set_title('Matriz de nulidad', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

**Informe interactivo con ydata-profiling** (opcional). Descomentar para regenerarlo. El HTML resultante puede pesar varios MB y demorar minutos:

```python
from ydata_profiling import ProfileReport
profile = ProfileReport(df, explorative=True, title="All The News 2.1")
profile.to_file("publications_data_profiling_report.html")
```

**Observaciones sobre calidad de datos:**
- `author`: ~38% faltante. No se usará para clasificación.
- `section`: ~34% faltante. Limita el análisis por sección.
- `article`: ~3.9% faltante. Filas sin cuerpo de artículo deben descartarse antes de modelar.
- `url` y `publication`: 141 faltantes cada uno (probablemente las mismas filas). Se excluyen al filtrar `publication`.
- `title`, `date`, `year`, `month`, `day`, `idx`, `article_idx`: completos.

In [ ]:
# Artículos por medio de prensa
articles_per_pub = df['publication'].value_counts()
print('Artículos por medio de prensa:')
print(articles_per_pub.to_string())

In [ ]:
counts_ordered = {lbl: articles_per_pub[label_to_pub[lbl]] for lbl in ORDER}

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(
    ORDER,
    [counts_ordered[lbl] for lbl in ORDER],
    color=[PALETTE[lbl] for lbl in ORDER],
    edgecolor='white', linewidth=0.8, width=0.55,
)
for bar, lbl in zip(bars, ORDER):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 60,
        f'{counts_ordered[lbl]:,}', ha='center', va='bottom',
        fontsize=13, fontweight='bold',
    )
ax.set_title('Cantidad de artículos por medio de prensa', fontsize=18, pad=15)
ax.set_xlabel('')
ax.set_ylabel('Cantidad de artículos', fontsize=14)
ax.tick_params(axis='both', labelsize=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.set_ylim(0, max(counts_ordered.values()) * 1.15)
ax.grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=0.8)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 medios con más artículos
top_5_publications = articles_per_pub.head(5).index
print('Top 5 medios:', top_5_publications.tolist())

df_top_5 = df[df['publication'].isin(top_5_publications)].copy()
df_top_5['publication_label'] = df_top_5['publication'].replace('The New York Times', 'NY Times')
print(f'\nDataFrame filtrado: {df_top_5.shape[0]:,} filas')
print(df_top_5['publication'].value_counts())

## B. Visualización temporal

In [ ]:
# Parsear fechas con formatos mixtos (ej. '2018-02-02' y '2019-05-23 09:30:04')
df_top_5['date_parsed'] = pd.to_datetime(df_top_5['date'], format='mixed', errors='coerce')

n_failed = df_top_5['date_parsed'].isnull().sum()
print(f'Fechas no parseadas: {n_failed}')
print(f'Rango temporal: {df_top_5["date_parsed"].min()} → {df_top_5["date_parsed"].max()}')

In [ ]:
df_top_5['year_month'] = df_top_5['date_parsed'].dt.to_period('M')

monthly_counts = (
    df_top_5.groupby(['year_month', 'publication_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=ORDER, fill_value=0)
)
monthly_counts.index = monthly_counts.index.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))
for lbl in ORDER:
    ax.plot(
        monthly_counts.index, monthly_counts[lbl],
        label=lbl, color=PALETTE[lbl], linewidth=1.8, alpha=0.9,
    )
ax.set_title('Artículos publicados por mes — top 5 medios de prensa', fontsize=18, pad=15)
ax.set_xlabel('')
ax.set_ylabel('Artículos por mes', fontsize=14)
ax.tick_params(axis='both', labelsize=13)
ax.legend(title='Publicación', bbox_to_anchor=(1.01, 1), loc='upper left',
          frameon=True, fontsize=12, title_fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=0.8)
fig.autofmt_xdate()
sns.despine()
plt.tight_layout()
plt.show()

**Observaciones temporales:**
- **Reuters** domina en volumen durante todo el período (2016-2020), con picos especialmente pronunciados en 2017-2018.
- **The New York Times** y **CNBC** tienen cobertura estable, pero con volumen considerablemente menor que Reuters.
- **The Hill** muestra actividad concentrada en ciertos períodos, posiblemente vinculada al ciclo político (elecciones).
- **People** tiene cobertura más uniforme y de bajo volumen mensual.
- Los picos de actividad en Reuters suelen coincidir con eventos de alta cobertura internacional (reportes financieros, eventos políticos globales).

## C. Limpieza de texto y conteo de palabras

In [ ]:
# La función `clean_text` se define en `src/utils.py` y se importa en la celda
# de importaciones. Se reproduce aquí su firma para referencia rápida:
#
#     def clean_text(df, column_name):
#         """Normaliza el texto: elimina dateline, pasa a minúsculas, quita
#         puntuación y números, y colapsa espacios."""
help(clean_text)

In [ ]:
# Aplicar clean_text sobre el cuerpo y el título.
df_top_5 = df_top_5.copy()
df_top_5['CleanText'] = clean_text(df_top_5, 'article')
df_top_5['CleanTextTitle'] = clean_text(df_top_5, 'title')

# Verificar resultado
print('=== Texto original (primer artículo Reuters) ===')
print(df_top_5[df_top_5['publication']=='Reuters']['article'].iloc[0][:300])
print()
print('=== Texto limpio ===')
print(df_top_5[df_top_5['publication']=='Reuters']['CleanText'].iloc[0][:300])

**Justificación de las transformaciones agregadas:**

- **Punto (`.`)**: separa oraciones pero no aporta información léxica.
- **Signo de exclamación (`!`) y pregunta ya estaba (` ?`)**: son puntuación estándar a eliminar.
- **Punto y coma (`;`) y paréntesis (`( )`)**: abundantes en texto periodístico, solo generan tokens vacíos.
- **Comillas (`"` `'`)**: frecuentes en citas directas; su eliminación no altera el vocabulario significativo.
- **Guion (`-`)**: aparece en palabras compuestas y datelines (ej. `reuters -`). Al reemplazarlo por espacio, las palabras compuestas quedan como tokens separados.
- **Slash (`/`) y backslash**: presentes en URLs y rutas que ya son indeseables.
- **Caracteres especiales restantes (`@`, `#`, `$`, etc.)**: no aportan valor léxico y pueden generar ruido.
- **Números (`\d+`)**: cifras como años, porcentajes y cantidades no son distintivas del medio y pueden sesgar el modelo.
- **Colapso de espacios**: resultado limpio tras todas las sustituciones anteriores.

## D. Elección de campos de texto

Los campos disponibles son `title` (título del artículo) y `article` (cuerpo del artículo).

**Análisis:**

| Campo | Ventajas | Desventajas |
|-------|----------|-------------|
| `article` (cuerpo) | Mucho más información, vocabulario más rico y distintivo por medio | Mayor longitud → mayor costo computacional; más ruido posible |
| `title` (título) | Breve, fácil de procesar, muy informativo en proporción | Muy corto → escasa señal estadística; mayor varianza |
| Combinación | Maximiza la información disponible | Puede generar desbalance si los títulos se repiten mucho respecto al cuerpo |

**Decisión:** trabajar principalmente con el **cuerpo del artículo** (`article`), ya que provee suficiente señal estadística para modelos de bolsa de palabras. En caso de artículos muy cortos (ej. briefs de Reuters), se puede considerar agregar el título concatenado para no desperdiciar información.

La columna `CleanText` construida a partir de `article` será la que se use en el resto del análisis.

## E. Pistas que identifican al medio de prensa

In [ ]:
# Búsqueda de auto-menciones en el cuerpo limpio, el título limpio y la URL en
# el texto crudo. Una mención cuenta si aparece en cualquiera de las tres fuentes.
mention_terms = {
    'Reuters': {
        'clean': ['reuters'],
        'url':   ['reuters.com'],
    },
    'The New York Times': {
        'clean': ['new york times', 'nytimes', 'nyt'],
        'url':   ['nytimes.com'],
    },
    'CNBC': {
        'clean': ['cnbc'],
        'url':   ['cnbc.com'],
    },
    'The Hill': {
        'clean': ['the hill', 'thehill'],
        'url':   ['thehill.com'],
    },
    'People': {
        'clean': ['people'],
        'url':   ['people.com'],
    },
}

print('Artículos con al menos una auto-mención del medio:\n')
rows = []
for pub in top_5_publications:
    sub = df_top_5[df_top_5['publication'] == pub]
    clean_terms = mention_terms[pub]['clean']
    url_terms = mention_terms[pub]['url']

    clean_pat = '|'.join(clean_terms) if clean_terms else None
    url_pat = '|'.join(url_terms) if url_terms else None

    in_article = (
        sub['CleanText'].fillna('').str.contains(clean_pat, case=False, na=False)
        if clean_pat else pd.Series(False, index=sub.index)
    )
    in_title = (
        sub['CleanTextTitle'].fillna('').str.contains(clean_pat, case=False, na=False)
        if clean_pat else pd.Series(False, index=sub.index)
    )
    in_url = (
        sub['article'].fillna('').str.contains(url_pat, case=False, na=False)
        if url_pat else pd.Series(False, index=sub.index)
    )
    has_mention = in_article | in_title | in_url
    n = int(has_mention.sum())
    total = len(sub)
    rows.append((pub, total, n, n / total * 100))
    print(f'  {pub}: {n:,}/{total:,} artículos ({n/total*100:.1f}%)')

mention_summary = pd.DataFrame(
    rows, columns=['Publicación', 'Total artículos', 'Con auto-mención', 'Porcentaje (%)']
).sort_values('Porcentaje (%)', ascending=False).reset_index(drop=True)
mention_summary

In [ ]:
# Ejemplos concretos de pistas en Reuters
reuters_samples = df_top_5[df_top_5['publication']=='Reuters']['article'].dropna().head(3)
for i, text in enumerate(reuters_samples):
    print(f'--- Reuters artículo {i+1} ---')
    print(text[:250])
    print()

In [ ]:
# Buscar patrones de dateline en Reuters (ej. 'Feb 2 (Reuters) -')
reuters_dateline = df_top_5[df_top_5['publication']=='Reuters']['article'].dropna()
has_dateline = reuters_dateline.str.contains(r'\(reuters\)', case=False, na=False)
print(f'Reuters: artículos con dateline "(Reuters)": {has_dateline.sum():,}/{len(reuters_dateline):,} ({has_dateline.mean()*100:.1f}%)')

# También buscar en NYT
nyt_text = df_top_5[df_top_5['publication']=='The New York Times']['article'].dropna()
has_nyt = nyt_text.str.contains(r'new york times', case=False, na=False)
print(f'NYT: artículos con mención "New York Times": {has_nyt.sum():,}/{len(nyt_text):,} ({has_nyt.mean()*100:.1f}%)')

**Pistas identificadas y decisión:**

- **Reuters**: la gran mayoría de artículos comienza con una firma de dateline en la forma `"Mes DD (Reuters) -"` (ej. `"Feb 2 (Reuters) -"`). Esta pista es extremadamente fuerte y haría trivial la clasificación. La función `clean_text` ya elimina el texto hasta el primer `\n`, lo que mitiga parcialmente este problema en artículos multilínea, pero en artículos cortos de un solo bloque la dateline persiste en el cuerpo.
- **CNBC/The Hill**: menor presencia de auto-menciones explícitas, pero pueden aparecer en URLs o frases de cierre.
- **People/NYT**: baja tasa de auto-menciones directas.

**Decisión:** conviene **eliminar o enmascarar** estas pistas para no construir un clasificador que simplemente detecte patrones triviales de autoría y no aprenda las diferencias reales de estilo y contenido. Una opción práctica es extender `clean_text` para reemplazar los nombres de medios conocidos por un token neutro (e.g., `[PUBLICATION]`), o directamente eliminarlos.

### Análisis de cobertura temporal por medio

Para la decisión de restringir por período temporal, se computa la ventana de cobertura común a todos los medios del top 5 (donde todos están activos) y la ventana común a todo el dataset.

In [ ]:
# Cobertura del top 5: ventana donde todos los medios tienen artículos publicados.
df_top_5['date_parsed'] = pd.to_datetime(
    df_top_5['date'], format='mixed', errors='raise'
)
coverage_top5 = (
    df_top_5.groupby('publication')['date_parsed']
    .agg(first='min', last='max')
    .sort_values('last')
)
common_start_top5 = coverage_top5['first'].max()
common_end_top5 = coverage_top5['last'].min()
print(f'Ventana común top 5: {common_start_top5.date()} → {common_end_top5.date()}')
print(coverage_top5.to_string())

In [ ]:
# Cobertura para el dataset completo (26 medios).
df['date_parsed'] = pd.to_datetime(df['date'], format='mixed', errors='raise')
coverage_all = (
    df.groupby('publication')['date_parsed']
    .agg(first='min', last='max')
    .sort_values('last')
)
common_start_all = coverage_all['first'].max()
common_end_all = coverage_all['last'].min()
print(f'Ventana común todos los medios: {common_start_all.date()} → {common_end_all.date()}')

# Tamaño de cada subconjunto bajo las ventanas comunes
mask_top5 = (df['date_parsed'] >= '2016-01-01') & (df['date_parsed'] <= '2019-12-01')
mask_all = (df['date_parsed'] >= '2017-05-01') & (df['date_parsed'] <= '2018-08-31')
n_top5 = df[mask_top5 & df['publication'].isin(top_5_publications)]['article'].count()
n_all = df[mask_all]['article'].count()
print(f'\nArtículos en ventana 2016-01 a 2019-12 (top 5): {n_top5:,}')
print(f'Artículos en ventana 2017-05 a 2018-08 (todos):  {n_all:,}')

## F. Restricción por sección o período temporal

**Análisis:**

El objetivo de la tarea es clasificar artículos **por medio de prensa**, no por tema. Sin embargo, si ciertos medios cubren ciertos temas con mucha más frecuencia que otros (ej. Reuters cubre mucho más finanzas y política internacional que People), el clasificador podría aprender a distinguir temas en lugar de estilos de escritura.

**Restricción por sección:**
- *Ventajas*: reduce el efecto del tema; permite evaluar si el estilo es distinguible incluso dentro de la misma sección temática.
- *Desventajas*: la columna `section` tiene ~34% de datos faltantes y no es consistente entre medios (cada medio tiene su propia taxonomía de secciones). Sería necesario alinear categorías manualmente.

**Restricción por período temporal:**
- *Ventajas*: reduce el sesgo por eventos históricos específicos que un medio pudo haber cubierto de manera diferente en distintos períodos.
- *Desventajas*: puede reducir drásticamente el volumen de datos, especialmente para medios con cobertura irregular. También puede introducir un sesgo temporal si el modelo luego se aplica a artículos de otros períodos.

**Conclusión:** para un análisis exploratorio inicial, no se restringe. Si se busca un clasificador más robusto y generalizable, restringir a una misma franja temporal (ej. 2018) y enmascarar los nombres de los medios sería lo más recomendable.

# Parte 2: Conteo de Palabras y Visualizaciones

## A. Palabras más frecuentes por medio

In [ ]:
# Grupo 1: NY Times · Reuters · The Hill
fig, axes = plt.subplots(1, 3, figsize=(16, 7))
plot_top_words(df_top_5, ORDER[:3], axes)
plt.suptitle(
    f'Top {N_TOP_WORDS} palabras más frecuentes (sin stopwords)',
    fontsize=16, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Grupo 2: CNBC · People
fig, axes = plt.subplots(1, 2, figsize=(12, 7))
plot_top_words(df_top_5, ORDER[3:], axes)
plt.suptitle(
    f'Top {N_TOP_WORDS} palabras más frecuentes (sin stopwords)',
    fontsize=16, y=1.01,
)
plt.tight_layout()
plt.show()

**Nubes de palabras por medio:**

Las nubes de palabras complementan los gráficos de barras al mostrar simultáneamente las palabras de mayor frecuencia para cada medio en una representación visual compacta.

In [ ]:
fig = plot_wordclouds_by_source(df_top_5, text_column='CleanText')
fig.suptitle('Nubes de palabras por medio de prensa', fontsize=16, y=1.02)
plt.show()

**Análisis de las palabras más frecuentes:**

- **Problema identificado:** palabras muy genéricas del periodismo ("said", "new", "also", "president", "trump") aparecen en el top de casi todos los medios, lo que dificulta identificar diferencias entre ellos. Para resaltar diferencias se podrían usar métricas que ponderen no solo qué tan frecuente es una palabra dentro de un medio, sino también qué tan exclusiva resulta respecto a los demás, priorizando palabras que son frecuentes en un medio pero raras en los otros.

- **Reuters** destaca en vocabulario financiero y de agencia: términos breves, técnicos, de alta densidad informativa.
- **The New York Times** y **CNBC** tienen mayor presencia de vocabulario político y económico.
- **The Hill** refleja cobertura política intensa.
- **People** tiene vocabulario diferenciado relacionado con cultura pop, entretenimiento y celebridades.

**Propuestas de mejora:**
1. Usar métricas de exclusividad relativa en lugar de frecuencia bruta, para destacar palabras distintivas de cada medio.
2. Considerar secuencias de palabras contiguas (dos o más palabras consecutivas como una sola unidad de análisis) para capturar frases características como "capitol hill" o "new york times".
3. Filtrar por sección temática para ver si las diferencias se mantienen dentro de un mismo tema.
4. Comparar la evolución de las palabras más frecuentes por fecha para detectar cambios de agenda.

## B. Medios con mayor cantidad de palabras

In [ ]:
# Contar palabras por artículo
df_top_5 = df_top_5.copy()
df_top_5['word_count'] = df_top_5['CleanText'].str.split().str.len()

# Estadísticas por publicación
word_stats = df_top_5.groupby('publication')['word_count'].agg(
    Media='mean', Mediana='median', Total='sum', Max='max'
).round(1).sort_values('Total', ascending=False)

print('Estadísticas de palabras por medio:')
print(word_stats.to_string())

In [ ]:
word_stats = (
    df_top_5.groupby('publication_label')['word_count']
    .agg(Media='mean', Mediana='median', Total='sum')
    .reindex(ORDER)
)
print(word_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel izquierdo: total de palabras
axes[0].bar(
    ORDER, word_stats['Total'],
    color=[PALETTE[lbl] for lbl in ORDER],
    edgecolor='white', width=0.55,
)
axes[0].set_title('Total de palabras por medio', fontsize=18, pad=15)
axes[0].set_xlabel('')
axes[0].set_ylabel('Total de palabras', fontsize=14)
axes[0].tick_params(axis='both', labelsize=13)
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{int(x):,}')
)
axes[0].grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=0.8)
sns.despine(ax=axes[0])

# Panel derecho: boxplot (estilo de referencia)
sns.boxplot(
    data=df_top_5,
    x='publication_label',
    y='word_count',
    order=ORDER,
    hue='publication_label',
    hue_order=ORDER,
    palette='viridis',
    legend=False,
    showfliers=False,
    width=0.5,
    linewidth=1.5,
    ax=axes[1],
)
axes[1].set_title('Longitud de artículos por medio de prensa', fontsize=18, pad=15)
axes[1].set_xlabel('')
axes[1].set_ylabel('Palabras por artículo', fontsize=14)
axes[1].tick_params(axis='both', labelsize=13)
axes[1].grid(True, axis='y', linestyle='--', linewidth=0.7, alpha=0.8)
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

**Observaciones:**

- **Reuters** tiene el mayor total de palabras por su alto volumen de artículos, pero sus artículos individuales tienden a ser **más cortos** (muchos son briefs de pocas líneas).
- **The New York Times** tiene artículos significativamente más extensos en promedio, típico de periodismo de largo aliento.
- Este desbalance en longitud puede ser un problema para modelos basados en conteos absolutos. Trabajar con frecuencias relativas —normalizadas por la longitud de cada artículo— mitiga este desequilibrio, al poner a los documentos en igualdad de condiciones independientemente de su extensión.
- Los artículos con `word_count` extremadamente alto (outliers) merecen revisión: pueden ser artículos concatenados, páginas completas descargadas, o errores en la extracción del texto.

## C. Matriz de menciones entre medios

In [ ]:
# Palabras clave de búsqueda para cada medio (en texto limpio/minúsculas)
pub_search_terms = {
    'Reuters':            'reuters',
    'The New York Times': 'new york times',
    'CNBC':               'cnbc',
    'The Hill':           'the hill',
    'People':             'people',   # nota: 'people' es ambiguo; se acepta el ruido
}

publications = list(top_5_publications)

# Construir matriz de menciones (i menciona a j)
mentions_matrix = pd.DataFrame(0, index=publications, columns=publications)

for source_pub in publications:
    source_texts = df_top_5[df_top_5['publication'] == source_pub]['CleanText'].fillna('')
    combined_text = ' '.join(source_texts)
    for target_pub in publications:
        if source_pub != target_pub:
            keyword = pub_search_terms[target_pub]
            # Conteo simple de ocurrencias del término
            count = combined_text.count(keyword)
            mentions_matrix.loc[source_pub, target_pub] = count

print('Matriz de menciones (fila i menciona a columna j):')
print(mentions_matrix.to_string())

In [ ]:
# Reindexar la matriz usando nombres cortos y orden canónico
display_matrix = mentions_matrix.copy()
display_matrix.index   = [pub_to_label[p] for p in display_matrix.index]
display_matrix.columns = [pub_to_label[p] for p in display_matrix.columns]
display_matrix = display_matrix.reindex(index=ORDER, columns=ORDER)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    display_matrix, annot=True, fmt='d', cmap='viridis',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Menciones', 'shrink': 0.8},
    annot_kws={'size': 13, 'weight': 'bold'},
)
ax.set_title('Matriz de menciones entre medios\n(fila = fuente, columna = mencionado)',
             fontsize=16, pad=12)
ax.set_xlabel('Medio mencionado', fontsize=14)
ax.set_ylabel('Medio que menciona', fontsize=14)
ax.tick_params(axis='both', labelsize=13)
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Construir grafo con nombres cortos
G = nx.DiGraph()
G.add_nodes_from(ORDER)

for lbl_i in ORDER:
    pub_i = label_to_pub[lbl_i]
    for lbl_j in ORDER:
        pub_j = label_to_pub[lbl_j]
        if pub_i != pub_j:
            w = int(mentions_matrix.loc[pub_i, pub_j])
            if w > 0:
                G.add_edge(lbl_i, lbl_j, weight=w)

# Normalización logarítmica: comprime el rango enorme (5–4769) a una escala visual legible
all_weights = np.array([G[u][v]['weight'] for u, v in G.edges()], dtype=float)
log_max = np.log1p(all_weights.max())

def log_width(w, min_w=0.6, max_w=9.0):
    return min_w + (max_w - min_w) * np.log1p(w) / log_max

RAD = 0.2
pos = nx.circular_layout(G, scale=1.3)
fig, ax = plt.subplots(figsize=(12, 10))

# ── Nodos ─────────────────────────────────────────────────────────────────────
nx.draw_networkx_nodes(
    G, pos, ax=ax,
    node_size=4500,
    node_color=[PALETTE[n] for n in G.nodes()],
    linewidths=1.8, edgecolors='#333333', alpha=0.95,
)
nx.draw_networkx_labels(
    G, pos, ax=ax,
    font_size=13, font_weight='bold', font_color='white',
)

# ── Aristas coloreadas por nodo fuente, grosor ∝ log(menciones) ───────────────
for source in ORDER:
    edges_src = [(u, v) for u, v in G.edges() if u == source]
    if not edges_src:
        continue
    widths = [log_width(G[u][v]['weight']) for u, v in edges_src]
    nx.draw_networkx_edges(
        G, pos, ax=ax,
        edgelist=edges_src,
        width=widths,
        edge_color=[PALETTE[source]] * len(edges_src),
        arrows=True, arrowsize=20,
        connectionstyle=f'arc3,rad={RAD}',
        min_source_margin=42, min_target_margin=42,
        alpha=0.85,
    )

# ── Leyenda de escala de grosor ───────────────────────────────────────────────
legend_vals = [10, 100, 1000]
legend_labels = ['~10', '~100', '~1 000']
for val, lbl in zip(legend_vals, legend_labels):
    ax.plot([], [], color='grey', linewidth=log_width(val), label=lbl, solid_capstyle='round')
leg = ax.legend(
    title='Menciones\n(escala log)',
    loc='lower left',
    frameon=True, fontsize=11, title_fontsize=11,
    handlelength=2.5,
)

ax.set_title(
    'Grafo de menciones entre medios de prensa\n(color ∝ origen · grosor ∝ log menciones)',
    fontsize=18, pad=15,
)
ax.axis('off')
plt.tight_layout()
plt.show()

**Observaciones sobre la matriz de menciones:**

- **Reuters** es el medio más mencionado por los demás, consistente con su rol de agencia de noticias internacional que provee material fuente para otros medios.
- El término **"People"** puede generar falsos positivos ya que es también una palabra común del inglés. Los conteos para este medio deben interpretarse con cautela.
- **The Hill** tiende a citar otros medios políticos (NYT, CNBC), coherente con su cobertura de análisis político.
- La ausencia de menciones en la diagonal confirma que se excluyeron correctamente las auto-menciones del cálculo.

## D. Preguntas propuestas

A continuación se proponen tres preguntas que podrían responderse a partir de estos datos:

---

### Pregunta 1: ¿Es posible identificar el medio de prensa únicamente a partir del vocabulario, sin usar pistas directas como el nombre del medio?

**Motivación:** La tarea de clasificación pierde valor si el modelo solo aprende a reconocer el nombre del medio en el texto. ¿Qué tan buena es la clasificación cuando se eliminan todas las menciones directas de los nombres de medios?

**Camino:** Enmascarar los nombres de los medios (y sus variantes) en `CleanText`, construir representaciones numéricas del texto que ponderen las palabras según su frecuencia dentro de cada artículo y su exclusividad respecto al resto del corpus, entrenar un clasificador y comparar la performance con y sin enmascaramiento. Una caída significativa indicaría que el modelo dependía de las pistas directas.

---

### Pregunta 2: ¿Difiere el estilo de escritura entre medios dentro de la misma sección temática?

**Motivación:** El clasificador podría estar aprendiendo diferencias temáticas (Reuters cubre finanzas, People cubre entretenimiento) más que diferencias de estilo. Si restringimos a artículos de la misma sección, ¿sigue siendo posible clasificar el medio?

**Camino:** Filtrar por sección (ej. solo "Politics" o "Business"), aplicar el mismo pipeline de clasificación, y analizar qué rasgos léxicos o estilísticos resultan más discriminativos dentro de esa sección.

---

### Pregunta 3: ¿Cómo evolucionó el vocabulario de cobertura de cada medio durante eventos de alta relevancia (ej. elecciones 2018, COVID-19 2020)?

**Motivación:** Entender si los medios convergen o divergen en su vocabulario durante eventos de alto impacto puede revelar diferencias editoriales.

**Camino:** Seleccionar ventanas temporales alrededor de eventos clave, calcular las palabras más distintivas de cada medio en esa ventana mediante alguna métrica que compare qué tan asociada está cada palabra a un medio frente a los demás, y comparar con un período de baja carga noticiosa equivalente. Una visualización del ranking de palabras a lo largo del tiempo permitiría mostrar la evolución de forma compacta.